# 1. 임베딩 (Embedding)

- 주로 자연어 처리 ( NLP )에서 사용되는 용어, 텍스트 데이터 내의 단어나 구(Phrase), 문장 등을 고정된 크기의 실수 벡터로 변환하는 과정 
- 컴퓨터는 텍스트를 수치적으로 분석하고 처리할 수 있게 된다. 
- 임베딩은 단어나 문장의 의미를 벡터 공간에 매핑(mapping)하여, 텍스트의 의미적, 문법적 속성을 반영하는 벡터로 변환하는 것을 목표로 한다. 

## 1.1 임베딩 과정 

1. tokenizer(토큰화)

- 원본텍스트 데이터는 토크노하 과정을 거쳐 개별 단어나 서브워드(subword) 단위로 나누어진다. 

2. Indexing(인덱싱)

- 토큰화된 토큰은 어위 사전(Vocabulary)에 따라 고유한 인덱스로 변환. 어휘 사전은 모델이 인식할 수 있는 모든 토큰을 포함하며, 각 토큰은 사전에서의 위치에 따라 고유한 정수 인덱스를 가진다. 

3. Embedding Lookup(임베딩 조회)

- 토큰 인덱스를 사용하여 임베딩 행렬에서 해당 토큰의 벡터를 조회, 임베딩 행렬은 일반적으로  
$||V| \times D| $형태
- 여기서 |V|는 어휘사전의 크기, D는 임베딩 벡터의 차원 

## 2.임베딩 층 만들기 

- 임베딩 층은 딥러닝 아키텍처에서 특정 도메인의 데이터를 저차원의 연속 벡터 공간으로 매핑하는 역할을 수행 
- 각 단어나 토큰을 고정된 크기의 밀집 벡터로변환하여, 단어간의 의미적 유사성을 벡터 공간에서의 거리로 나타내는 기능을 한다. 딥러닝 모델 내에서 임베딩층은 초기 입력 데이터의 차원을 축소시키고, 연산을 효율적으로 만들어주는 중요한 구성 요소 

In [ ]:
import torch
import torch.nn as nn

# 임베딩 층을 생성
embedding_layer = nn.Embedding(num_embeddings=10, embedding_dim=3)
print(embedding_layer)

# 임의의 정수 인덱스로 구성된 입력 데이터 (예: 단어 인덱스)
input_data = torch.LongTensor([0, 1,])

# 입력 데이터를 임베딩 층에 전달하여 임베딩 벡터를 얻습니다.
embedded_data = embedding_layer(input_data)
print(embedded_data)

## 3. Word2Vec

- 단어 임베딩을 위한 모델, 2013 구글에서 개발
- 단어 벡터 공간에 매핑하여 단어 간의 의미적 관계를 수치적으로 표현하는 모델
- 주요 목적은 대규모 텍스트 코퍼스 안에서 단어의 의미를 포착하여 저차원의 연속적인 벡터로 변환 
- 단어의 의미적 유사성을 반영하여, 이를 임베딩 벡터 또는 단어 임베딩이라고 함

### 3.1 Word2Vec 이론

- CBOW(Continuous Bag of Words )
    - 주변 단어들을 사용하여 중간에 위치한 타켓 단어를 예측 
    - 예를들어, "the cat sits on the"에서 주변 단어들을 통해 "mat"을 예측하는 방식 
    - 맥락 내의 여러 단어로부터 타켓 단어를 예측하기 때문에 작은 데이터셋에서 잘 작동 

- Skip-Gram
    - 타켓 단어로부터 주변의 맥락 단어들을 예측 
    - 예를들어, "cat"이 주어졌을 때 "the", "sits", "on", "the" 등을 예측
    - Skip-Gram은 각 타켓 단어로부터 주변 단어들을 독립적으로 예측하기 때문에 CBOW보다 큰 데이터셋에서 더 좋은 성능을 보임

In [2]:
!uv pip install gensim
import gensim.downloader as api

list(api.info()['models'].keys())

Using Python 3.12.12 environment at: /Users/donghun2/workspace/machine_learning/dacon/.venv
Resolved 5 packages in 72ms                                          
Installed 2 packages in 4ms                                 
 + gensim==4.4.0
 + smart-open==8.0.1


['fasttext-wiki-news-subwords-300',
 'conceptnet-numberbatch-17-06-300',
 'word2vec-ruscorpora-300',
 'word2vec-google-news-300',
 'glove-wiki-gigaword-50',
 'glove-wiki-gigaword-100',
 'glove-wiki-gigaword-200',
 'glove-wiki-gigaword-300',
 'glove-twitter-25',
 'glove-twitter-50',
 'glove-twitter-100',
 'glove-twitter-200',
 '__testing_word2vec-matrix-synopsis']

In [3]:
import torch
import torch.nn as nn
from gensim.models import Word2Vec

# 샘플 문장
sentences = [
    ['this', 'is', 'the', 'first', 'sentence', 'for', 'word2vec'],
    ['this', 'is', 'the', 'second', 'sentence'],
    ['yet', 'another', 'sentence'],
    ['one', 'more', 'sentence'],
    ['and', 'the', 'final', 'sentence'],
]

# Word2Vec 모델 학습
model_wv = Word2Vec(sentences, vector_size=5, window=5, min_count=1, workers=4)

# 임베딩 매트릭스 크기 설정
num_embeddings = len(model_wv.wv.index_to_key)
embedding_dim = model_wv.vector_size

# 임베딩 매트릭스 만들기
wv_matrix = torch.FloatTensor([model_wv.wv[word] for word in model_wv.wv.index_to_key])

# 단어와 인덱스 매핑 확인
word_index_mapping = model_wv.wv.key_to_index
print('----단어사전----')
for word, index in word_index_mapping.items():
    print(f"{index}: {word}")

print('----임베딩 메트릭스----\n', wv_matrix)
print('임베딩 메트릭스 크기:', wv_matrix.shape)

# PyTorch 임베딩 레이어에 사전 훈련된 가중치를 사용
w2v_embedding_layer = nn.Embedding.from_pretrained(wv_matrix, freeze=False)
print(w2v_embedding_layer)

input_data = torch.LongTensor([0, 1]) # [sentence, the]

# 입력 데이터를 임베딩 층에 전달하여 임베딩 벡터를 얻습니다.
embedded_data = w2v_embedding_layer(input_data)
print(embedded_data)

----단어사전----
0: sentence
1: the
2: is
3: this
4: final
5: and
6: more
7: one
8: another
9: yet
10: second
11: word2vec
12: for
13: first
----임베딩 메트릭스----
 tensor([[-0.0107,  0.0047,  0.1021,  0.1802, -0.1861],
        [-0.1423,  0.1292,  0.1795, -0.1003, -0.0753],
        [ 0.1476, -0.0307, -0.0907,  0.1311, -0.0972],
        [-0.0363,  0.0575,  0.0198, -0.1657, -0.1890],
        [ 0.1462,  0.1014,  0.1352,  0.0153,  0.1270],
        [-0.0681, -0.0189,  0.1154, -0.1504, -0.0787],
        [-0.1502, -0.0186,  0.1908, -0.1464, -0.0467],
        [-0.0388,  0.1615, -0.1186,  0.0009, -0.0951],
        [-0.1921,  0.1001, -0.1752, -0.0878, -0.0007],
        [-0.0059, -0.1532,  0.1923,  0.0996,  0.1847],
        [-0.1632,  0.0899, -0.0827,  0.0165,  0.1700],
        [-0.0892,  0.0904, -0.1357, -0.0710,  0.1880],
        [-0.0316,  0.0064, -0.0828, -0.1537, -0.0302],
        [ 0.0494, -0.0178,  0.1107, -0.0549,  0.0452]])
임베딩 메트릭스 크기: torch.Size([14, 5])
Embedding(14, 5)
tensor([[-0.0107,  0.004

/var/folders/ct/csn7gw6j4lzg64vvct28bvj00000gn/T/ipykernel_31665/1408944856.py:22: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /Users/runner/work/pytorch/pytorch/torch/csrc/utils/tensor_new.cpp:255.)
  wv_matrix = torch.FloatTensor([model_wv.wv[word] for word in model_wv.wv.index_to_key])


- [gensim?](../tip/gensim.md). 

In [4]:
import torch
import torch.nn as nn
import gensim.downloader as api

# word2vec-ruscorpora-300 모델 로드
wv_vectors = api.load('word2vec-ruscorpora-300')

# 어휘 크기와 임베딩 차원을 가져옵니다.
vocab_size = len(wv_vectors.key_to_index)
embedding_dim = wv_vectors.vector_size

# 임베딩 행렬을 준비합니다.
weights = torch.FloatTensor(wv_vectors.vectors)

wv_embedding_layer = nn.Embedding.from_pretrained(weights, freeze=False)
print(wv_embedding_layer)

[==================================================] 100.0% 198.8/198.8MB downloaded
Embedding(184973, 300)


In [5]:
# 임의의 정수 인덱스로 구성된 입력 데이터 (예: 단어 인덱스)
input_data = torch.LongTensor([0, 1])

# 입력 데이터를 임베딩 층에 전달하여 임베딩 벡터를 얻습니다.
embedded_data = wv_embedding_layer(input_data)
print(embedded_data)

tensor([[-0.0192, -0.0443,  0.0114,  0.1661, -0.0472, -0.0196, -0.0083,  0.0136,
         -0.0684,  0.0513,  0.0114,  0.0386, -0.0284,  0.0114, -0.0111, -0.0080,
          0.0277,  0.0066, -0.0332, -0.0141,  0.0754,  0.0658, -0.0895,  0.0482,
         -0.0290, -0.0866,  0.0163,  0.0382,  0.0271,  0.0176,  0.0340, -0.0510,
          0.0663, -0.0745, -0.0551,  0.0317,  0.0791, -0.0235,  0.0255,  0.0115,
         -0.0044,  0.0191,  0.0543, -0.0925,  0.0692, -0.1207,  0.0221, -0.0576,
          0.0216, -0.1466,  0.0589,  0.0186,  0.0293, -0.0094,  0.1003, -0.0417,
         -0.1334,  0.0646,  0.0214, -0.0021, -0.0357,  0.0387, -0.0715, -0.0108,
          0.0396,  0.0073, -0.0769,  0.0347, -0.0592,  0.0878, -0.0556,  0.0112,
         -0.0526, -0.0820, -0.0115, -0.0097,  0.0682, -0.0351,  0.0304,  0.0024,
         -0.0008, -0.0474, -0.0720, -0.0846,  0.0519,  0.0203,  0.0096,  0.0250,
         -0.0680, -0.0594,  0.0552,  0.0155, -0.0710, -0.0464, -0.0277, -0.0523,
          0.0651, -0.0205,  